In [ ]:
exams_list = [
    "Auditor-Fiscal do Trabalho - MTE - 2003 - Prova 3",
    "Auditor-Fiscal do Trabalho - MTE - 2006 - Prova 2",
    "Auditor-Fiscal do Trabalho - MTE - 2010 - Prova 2",
]

In [ ]:
# Start of the block of each exam in the answer key PDF (it has the keys of all the exams of the year)
answer_key_blocks = {
    "Auditor-Fiscal do Trabalho - MTE - 2003 - Prova 3": r'^Prova 3$',
    "Auditor-Fiscal do Trabalho - MTE - 2006 - Prova 2": r'^Prova 2$',
    # 2010: Prova 2, Gabarito 1 (the exam pages say 'Prova 2 - Gabarito 1')
    "Auditor-Fiscal do Trabalho - MTE - 2010 - Prova 2": r'^Prova 2 - .*\nPPrroovvaa ddee GGaabbaarriittoo 11.*$',
}

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import pdfplumber
from utils.utils_oab import extract_by_columns, join_texts, load_rules, clean_text
from utils.utils_mte import extract_section, split_questions, extract_questions, extract_answer_key

## Prova

In [ ]:
idx = 0
exam_path = f'all_exams_mte/{exams_list[idx]}-prova.pdf'
answer_key_path = f'all_exams_mte/{exams_list[idx]}-gabarito.pdf'
print(f'Prova: {exams_list[idx]}')

In [ ]:
with pdfplumber.open(exam_path) as pdf:
    size_pdf = len(pdf.pages)
dict_mid = {i: 0.5 for i in range(size_pdf)}
# Column split (fraction of the page width) of the pages whose left column goes past the middle
custom_mid = {
    "Auditor-Fiscal do Trabalho - MTE - 2003 - Prova 3": {i: 0.509 for i in range(1, 14)},
}
dict_mid.update(custom_mid.get(exams_list[idx], {}))
literals, patterns = load_rules('extraction_config/cleaning_rules_mte.json')
question_literals, question_patterns = load_rules('extraction_config/cleaning_rules_mte.json', stage='question')

## Texto da prova

In [ ]:
texts_left, texts_right = extract_by_columns(exam_path, dict_mid)
text = join_texts(texts_left, texts_right)

In [ ]:
text = clean_text(text, literals, patterns)

## Seção DIREITO DO TRABALHO

In [ ]:
section = extract_section(text)
print(section[:1000])

In [ ]:
print(section[-1000:])

## Questões e opções

In [ ]:
questions_split = split_questions(section)
print(f'{len(questions_split)} questions: {min(questions_split)} to {max(questions_split)}')

In [ ]:
extracted_questions = extract_questions(questions_split, question_literals, question_patterns)

In [ ]:
extracted_questions[0]

## Gabarito

In [ ]:
import json

answer_key = extract_answer_key(answer_key_path, answer_key_blocks[exams_list[idx]])
print(f'Answer key: {len(answer_key)} answers')
# Answers of the final answer key where the answer key PDF is the preliminary one (the reason is in the note)
with open('extraction_config/answer_key_corrections.json', encoding='utf-8') as f:
    corrections = json.load(f).get(exams_list[idx], {})
for number, correction in corrections.items():
    print(f'Answer of question {number}: {answer_key.get(int(number))} -> {correction["answer"]}')
    answer_key[int(number)] = correction['answer']

In [ ]:
# Annulled questions (without answer in the answer key) are not saved
annulled = [q['question_number'] for q in extracted_questions if answer_key.get(q['question_number']) is None]
extracted_questions = [q for q in extracted_questions if q['question_number'] not in annulled]
print(f'Annulled questions removed: {len(annulled)} {annulled}')
for question_data in extracted_questions:
    question_data['answer'] = answer_key[question_data['question_number']]

## Salvar

In [ ]:
df_exam = pd.DataFrame([{
    'question_number': question_data['question_number'],
    'question': question_data['question'],
    'options': list(question_data['options'].values()),
    'exam': exams_list[idx],
    'answer': question_data['answer'],
} for question_data in extracted_questions])
df_exam.head()

In [ ]:
# Replaces the rows of this exam in the CSV, without duplicating
csv_path = 'csvs/mte/mte_esaf_questions_all.csv'
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
if os.path.exists(csv_path):
    df_saved = pd.read_csv(csv_path, encoding='utf-8-sig')
    df_saved = df_saved[df_saved['exam'] != exams_list[idx]]
    df_exam = pd.concat([df_saved, df_exam], ignore_index=True)
df_exam.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'{csv_path}: {len(df_exam)} questions from {df_exam["exam"].nunique()} exams')